# Data Access Layer — Manual Testing

This notebook tests the data access layer against simulated case data.

**Prerequisites:**
- `pip install -r requirements.txt`
- Generate data: `python -m data --output data/simulated/ --seed 42 --cases 5`

## 1. Setup — Load Gateway & Catalog

In [ ]:
import sys
sys.path.insert(0, '..')

from data.gateway import SimulatedDataGateway
from data.catalog import DataCatalog
from tools.data_tools import init_tools, list_available_tables, get_table_schema, query_table

# Load from generated case folders
gw = SimulatedDataGateway.from_case_folders('../data/simulated')
catalog = DataCatalog()
init_tools(gw, catalog)

print('Gateway and catalog loaded.')

## 2. List Available Cases

In [ ]:
cases = gw.list_case_ids()
print(f'Available cases ({len(cases)}): {cases}')

## 3. Select a Case & List Tables

In [ ]:
gw.set_case('CASE-00001')
print(list_available_tables())

## 4. Inspect Table Schema

In [ ]:
import json

schema = get_table_schema('bureau_full')
print(json.dumps(json.loads(schema), indent=2))

In [ ]:
schema = get_table_schema('cross_bu')
print(json.dumps(json.loads(schema), indent=2))

## 5. Query Bureau Data

In [ ]:
print('=== Bureau Full (first 3 rows) ===')
print(query_table('bureau_full', limit=3))

## 6. Query Cross-BU Cards

In [ ]:
print('=== All Cards for CASE-00001 ===')
print(query_table('cross_bu'))

## 7. Filter — Find Delinquent Cards

In [ ]:
print('=== Delinquent Cards (90dpb) ===')
print(query_table('cross_bu', filter_column='account_status', filter_value='90dpb'))

print()
print('=== Delinquent Cards (30dpb) ===')
print(query_table('cross_bu', filter_column='account_status', filter_value='30dpb'))

## 8. Query Spend Transactions

In [ ]:
print('=== Spend Transactions (first 5) ===')
print(query_table('pmts_detail', limit=5))

## 9. Query Model Scores

In [ ]:
print('=== Model Scores (first row, first 20 fields) ===')
result = query_table('model_scores', limit=1)
data = json.loads(result)
if data:
    row = data[0]
    for i, (k, v) in enumerate(row.items()):
        if i >= 20:
            print(f'  ... and {len(row) - 20} more fields')
            break
        print(f'  {k}: {v}')

## 10. Query Score Drivers

In [ ]:
print('=== Score Drivers ===')
print(query_table('score_drivers', limit=5))

## 11. Query Other Tables

In [ ]:
print('=== Customer Tenure ===')
print(query_table('cust_tenure'))

print()
print('=== Income & DTI ===')
print(query_table('income_dti'))

print()
print('=== WCC Flags ===')
print(query_table('wcc_flags'))

print()
print('=== XBU Summary ===')
print(query_table('xbu_summary'))

## 12. Edge Cases

In [ ]:
# Missing table
print('=== Missing Table ===')
print(query_table('nonexistent_table'))

print()
# Filter with no matches
print('=== Filter with No Matches ===')
print(query_table('cross_bu', filter_column='account_status', filter_value='999dpb'))

## 13. Switch Case & Compare

In [ ]:
# Compare bureau data across two cases
for case_id in ['CASE-00001', 'CASE-00002']:
    gw.set_case(case_id)
    result = query_table('bureau_full', limit=1)
    data = json.loads(result)
    if data:
        row = data[0]
        print(f'\n=== {case_id} ===')
        print(f'  FICO: {row.get("fico_score", "N/A")}')
        print(f'  SBFE: {row.get("sbfe_score", "N/A")}')
        print(f'  Delinquent Trades: {row.get("delinquent_external_trades", "N/A")}')
        print(f'  External Exposure: ${row.get("overall_external_exposure", "N/A")}')
        print(f'  Judgements: {row.get("judgements_org_count", "N/A")}')
        print(f'  Liens: {row.get("lien_org_count", "N/A")}')

## 14. Full Data Catalog

In [ ]:
print(catalog.to_prompt_context())